In [39]:
import timeit
from ppg_basis import ppgGenerator
from ppg_constants import basis_types
import matplotlib.pyplot as plt
import os

print(timeit)
print(getattr(timeit, "__file__", "no __file__"))

<module 'timeit' from 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\\Lib\\timeit.py'>
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\timeit.py


In [40]:
def gen_pipeline(x, L):
    return x + L

print(timeit.timeit(lambda: gen_pipeline(1, 2), number=1))

2.1999876480549574e-06


In [ ]:
# constants

Ls = [2, 3, 4, 5, 10]

def gen_pipeline(duration: int, L: int, basis_type: str, solver: str):
    print(type(ppgGenerator))
    ppgGen = ppgGenerator(fs=125,
                          hr=60,
                          mu=0.85,
                          sigma=0.05,
                          duration=duration,
                          L= L,
                          basis_type=basis_type,
                          solver = solver)
    return ppgGen.generate_signal()

durations = [10, 25, 50, 75, 100, 250, 500, 750, 1000]

# FIGURE TOP

In [46]:
def ode_vs_base_analysis():
    """
    Returns an array containing:
    [
        [
            RK3_base for gaussian,
            RK4_base for gaussian
        ],
        [
            RK3_base for gamma,
            RK4_base for gamma
        ],
        [
            RK3_base for skewed-gaussian,
            RK4_base for skewed-gaussian
        ]
    ]
    """
    duration = 10
    toReturn = []
    for basis_type in basis_types:
        RK3_base = []
        RK4_base = []
        for L in Ls:
            RK3_time = timeit.timeit(lambda : gen_pipeline(duration, L, basis_type, 'rk3'), number = 5)
            RK4_time = timeit.timeit(lambda : gen_pipeline(duration, L, basis_type, 'rk4'), number = 5)
            base_time = timeit.timeit(lambda : gen_pipeline(duration, L, basis_type, 'basis'), number = 5)
            RK3_base.append(RK3_time / base_time)
            RK4_base.append(RK4_time / base_time)
        toReturn.append((RK3_base, RK4_base))
    return toReturn

top_pipeline_values = ode_vs_base_analysis()

True


TypeError: 'module' object is not callable

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for i, (d1, d2) in enumerate(top_pipeline_values):
    axes[i].hist(d1, bins=15, alpha=0.6, label="data{}_1".format(i+1))
    axes[i].hist(d2, bins=15, alpha=0.6, label="data{}_2".format(i+1))
    axes[i].set_title(f"Histogram {i+1}")
    axes[i].legend()

plt.tight_layout()
plt.show()

# FIGURE BOTTOM

In [ ]:
def fft_template_vs_base_analysis(basis_type: str):
    """
    given a basis_type (a graph in row 2), returns an array of the following:
    [
        [ # template_arr
            [ # for L = 2, 3, ...
                template_time / base_time for duration = 10, 25, ...
            ],
        ],
        [ # fft_arr
            [ # for L = 2, 3, ...
                template_time / base_time for duration = 10, 25, ...
            ],
        ]
    ]
    """
    template_arr = []
    fft_arr = []
    for L in Ls:
        L_template = []
        L_fft = []
        for duration in durations:
            template_time = timeit.timeit(lambda : gen_pipeline(duration, L, basis_type, 'template'))
            fft_time = timeit.timeit(lambda : gen_pipeline(duration, L, basis_type, 'fft'))
            base_time = timeit.timeit(lambda : gen_pipeline(duration, L, basis_type, 'basis'))
            L_template.append(template_time / base_time)
            L_fft.append(fft_time / base_time)
        template_arr.append(L_template)
        fft_arr.append(L_fft)
    return [template_arr, fft_arr]

bottom_pipeline_values = [
    fft_template_vs_base_analysis('gaussian'),
    fft_template_vs_base_analysis('gamma'),
    fft_template_vs_base_analysis('skewed-gaussian')
]